In [0]:
dbutils.widgets.text("p_environment", "production")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../Includes/configuration"

In [0]:
%run "../Includes/common_functions"

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

In [0]:
movie_cast_schema = StructType(fields=[
  StructField('movieId', IntegerType(), True),
  StructField('personId', IntegerType(), True),
  StructField('characterName', StringType(), True),
  StructField('genderId', IntegerType(), True),
  StructField('castOrder', IntegerType(), True)
])

In [0]:
movie_cast_df = spark.read \
                .schema(movie_cast_schema) \
                .option("multiline", True) \
                .json(f'{bronze_folder_path}/{v_file_date}/movie_cast.json')

In [0]:
movie_cast_df.printSchema

<bound method DataFrame.printSchema of DataFrame[movieId: int, personId: int, characterName: string, genderId: int, castOrder: int]>

In [0]:
display(movie_cast_df)

movieId,personId,characterName,genderId,castOrder
13673,7427,Merry,1,13
13673,15031,Aubie,0,14
13673,1599175,Officer Staff,2,15
13673,95875,Husband,2,16
13836,18918,Jack Bruno,2,0
13836,1285,Sara,1,1
13836,23498,Seth,2,2
13836,17832,Dr. Alex Friedman,1,3
13836,59238,Pope,2,4
13836,53261,Tina,1,5


In [0]:
from pyspark.sql.functions import col, current_timestamp, lit

In [0]:
movie_cast_dropped_df = movie_cast_df.drop(col("genderId"), col("castOrder"))

In [0]:
movie_cast_final_df = add_ingestion_date(movie_cast_dropped_df) \
                .withColumnsRenamed({"movieId": "movie_Id", "personId": "person_Id", "characterName": "character_Name"}) \
                .withColumn("environment", lit(v_environment)) \
                .withColumn("file_date", lit(v_file_date))

In [0]:
display(movie_cast_final_df)

movie_Id,person_Id,character_Name,ingestion_date,environment,file_date
13673,7427,Merry,2026-09-11T04:24:17.862279Z,production,2024-12-30
13673,15031,Aubie,2026-09-11T04:24:17.862279Z,production,2024-12-30
13673,1599175,Officer Staff,2026-09-11T04:24:17.862279Z,production,2024-12-30
13673,95875,Husband,2026-09-11T04:24:17.862279Z,production,2024-12-30
13836,18918,Jack Bruno,2026-09-11T04:24:17.862279Z,production,2024-12-30
13836,1285,Sara,2026-09-11T04:24:17.862279Z,production,2024-12-30
13836,23498,Seth,2026-09-11T04:24:17.862279Z,production,2024-12-30
13836,17832,Dr. Alex Friedman,2026-09-11T04:24:17.862279Z,production,2024-12-30
13836,59238,Pope,2026-09-11T04:24:17.862279Z,production,2024-12-30
13836,53261,Tina,2026-09-11T04:24:17.862279Z,production,2024-12-30


In [0]:
# overwrite_partition("movie_silver", "movie_cast", "file_date", v_file_date)

In [0]:
merge_delta_lake_2(movie_cast_final_df, "movie_silver", "movie_cast", "movie_Id", "person_Id", "file_date")

In [0]:
# movie_cast_final_df.write.mode("append").partitionBy("file_date").format("delta").saveAsTable("movie_silver.movie_cast")

In [0]:
display(spark.read.table("movie_silver.movie_cast"))

movie_Id,person_Id,character_Name,ingestion_date,environment,file_date
285,85,Captain Jack Sparrow,2026-09-11T03:39:17.407883Z,production,2024-12-16
285,114,Will Turner,2026-09-11T03:39:17.407883Z,production,2024-12-16
285,116,Elizabeth Swann,2026-09-11T03:39:17.407883Z,production,2024-12-16
285,1640,William Bootstrap Bill Turner,2026-09-11T03:39:17.407883Z,production,2024-12-16
285,1619,Captain Sao Feng,2026-09-11T03:39:17.407883Z,production,2024-12-16
285,2440,Captain Davy Jones,2026-09-11T03:39:17.407883Z,production,2024-12-16
285,118,Captain Hector Barbossa,2026-09-11T03:39:17.407883Z,production,2024-12-16
285,1709,Admiral James Norrington,2026-09-11T03:39:17.407883Z,production,2024-12-16
285,2449,Joshamee Gibbs,2026-09-11T03:39:17.407883Z,production,2024-12-16
285,2441,Lord Cutler Beckett,2026-09-11T03:39:17.407883Z,production,2024-12-16


In [0]:
%sql
SELECT file_date, COUNT(1)
FROM movie_silver.movie_cast
GROUP BY file_date;

file_date,count(1)
2024-12-16,30000
2024-12-23,15000
2024-12-30,5000


In [0]:
%sql
DESCRIBE EXTENDED movie_silver.movie_cast;

col_name,data_type,comment
movie_Id,int,null
person_Id,int,null
character_Name,string,null
ingestion_date,timestamp,null
environment,string,null
file_date,string,null
# Partition Information,,
# col_name,data_type,comment
file_date,string,null
,,


In [0]:
dbutils.notebook.exit("Success")